In [3]:
import pandas as pd

In [8]:
df = pd.read_csv('C:/Users/stepa/Downloads/wage.csv')

In [9]:
df

,person_id,gender,wage
0,0,1,46793.603811
1,1,1,33481.575720
2,2,1,44523.699084
3,3,1,15995.576829
4,4,0,10282.631224
...,...,...,...
1001,995,1,66503.737185
1002,996,1,9972.956272
1003,997,0,104504.616392
1004,998,1,98927.903076


In [14]:
df.gender = df.gender.apply(lambda x: 'F' if x == 0 else 'M')

In [15]:
df

,person_id,gender,wage
0,0,M,46793.603811
1,1,M,33481.575720
2,2,M,44523.699084
3,3,M,15995.576829
4,4,F,10282.631224
...,...,...,...
1001,995,M,66503.737185
1002,996,M,9972.956272
1003,997,F,104504.616392
1004,998,M,98927.903076


In [20]:
df.groupby('gender', as_index = False)['wage'].mean()

,gender,wage
0,F,40855.747261
1,M,46815.944005


In [35]:
duplicates = df.person_id.value_counts()
duplicates = duplicates[duplicates > 1]
duplicates

person_id
12    2
13    2
14    2
15    2
16    2
17    2
Name: count, dtype: int64

In [44]:
df.groupby('person_id', as_index = False)['wage'].nunique()[12:18]

,person_id,wage
12,12,1
13,13,1
14,14,1
15,15,1
16,16,1
17,17,1


In [53]:
df = df.drop_duplicates(subset='person_id')

In [60]:
df['person_id'].value_counts().max()

np.int64(1)

In [61]:
df.wage.describe()

count      1000.000000
mean      43694.227404
std       55352.539343
min     -287418.645743
25%       14489.682367
50%       27309.529498
75%       52021.080258
max      755320.874132
Name: wage, dtype: float64

In [62]:
df = df[df['wage'] > 0]

In [63]:
df.wage.describe()

count       995.000000
mean      44306.969585
std       54302.194392
min         945.648458
25%       14683.306148
50%       27519.361794
75%       52267.313664
max      755320.874132
Name: wage, dtype: float64

In [68]:
bonus = pd.read_csv('C:/Users/stepa/Downloads/bonus.csv', sep = ';')

In [94]:
bonus

,person_id,bonus
0,905,85059.638382
1,836,7703.346074
2,287,3120.269742
3,548,5347.987142
4,575,137257.490614
...,...,...
445,488,28102.252903
446,913,55549.347647
447,616,620397.407705
448,110,14086.067488


8. Чтобы посчитать итоговую зарплату, нам нужно по каждому человеку знать и оклад, и премию. Для этого надо будет соединить (сджойнить) таблицы по `person_id`. Используйте для этого функцию `pd.merge`. Помните, что параметр `how` должен быть `'outer'`, чтобы сохранить те записи, что есть только в одной таблице. Результат запишите в новый dataframe `df`

In [120]:
salary = pd.merge(df, bonus, how = 'outer', on = 'person_id')

In [121]:
salary

,person_id,gender,wage,bonus
0,0,M,46793.603811,3.332934e+04
1,1,M,33481.575720,NaN
2,2,M,44523.699084,3.192912e+06
3,3,M,15995.576829,2.196858e+04
4,4,F,10282.631224,NaN
...,...,...,...,...
994,995,M,66503.737185,3.452137e+03
995,996,M,9972.956272,3.892790e+05
996,997,F,104504.616392,5.380978e+04
997,998,M,98927.903076,NaN


9. Наконец, давайте посчитаем итоговую зарплату
    1. Замените отсутствующие записи в колонке `bonus` нулями
    1. Уберите людей без `wage` - это те "плохие" записи, от которых мы избавлялись на предыдущих шагах
    1. Сделайте новую колонку `total`, которая будет равна 12 окладам и премии
    1. Посчитайте среднюю и медианную итоговую зарплату в разрезе по полу. Подсказка: вместо функции агрегации можно написать `.agg()` и перечислить внутри нужные агрегаты

In [122]:
salary['bonus'] = salary['bonus'].fillna(0)

In [123]:
salary[salary['wage'].isna()]

,person_id,gender,wage,bonus
29,28,NaN,NaN,17095.225199
42,43,NaN,NaN,268778.439804


In [124]:
salary.wage.describe()

count       997.000000
mean      44255.182604
std       54260.428463
min         945.648458
25%       14677.625229
50%       27365.332097
75%       52042.472920
max      755320.874132
Name: wage, dtype: float64

In [126]:
salary = salary.dropna()

In [128]:
salary

,person_id,gender,wage,bonus
0,0,M,46793.603811,3.332934e+04
1,1,M,33481.575720,0.000000e+00
2,2,M,44523.699084,3.192912e+06
3,3,M,15995.576829,2.196858e+04
4,4,F,10282.631224,0.000000e+00
...,...,...,...,...
994,995,M,66503.737185,3.452137e+03
995,996,M,9972.956272,3.892790e+05
996,997,F,104504.616392,5.380978e+04
997,998,M,98927.903076,0.000000e+00


In [131]:
salary.loc[:, 'total'] = 12 * salary['wage'] + salary['bonus']

In [132]:
salary

,person_id,gender,wage,bonus,total
0,0,M,46793.603811,3.332934e+04,5.948526e+05
1,1,M,33481.575720,0.000000e+00,4.017789e+05
2,2,M,44523.699084,3.192912e+06,3.727197e+06
3,3,M,15995.576829,2.196858e+04,2.139155e+05
4,4,F,10282.631224,0.000000e+00,1.233916e+05
...,...,...,...,...,...
994,995,M,66503.737185,3.452137e+03,8.014970e+05
995,996,M,9972.956272,3.892790e+05,5.089544e+05
996,997,F,104504.616392,5.380978e+04,1.307865e+06
997,998,M,98927.903076,0.000000e+00,1.187135e+06


In [133]:
salary.isna().sum()

person_id    0
gender       0
wage         0
bonus        0
total        0
dtype: int64

10. Сохраните `df` в файл, используя метод `to_csv()`. Не записывайте индексы

In [134]:
salary.to_csv('salary.csv', index=False)